In [ ]:
import os, zipfile, gzip, shutil
import numpy as np
import pandas as pd
from io import BytesIO
from itertools import combinations, permutations
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from google.colab import files

In [ ]:
# PARAMETERS
CUTOFF = 4.0
TOP_PERCENT = 0.20
OUT_DIR = "/content/simple_residue_results"



os.makedirs(OUT_DIR, exist_ok=True)
parser = PDBParser(QUIET=True)

def upload_zip(prompt_text):
    print(prompt_text)
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise SystemExit("No file uploaded.")
    # take the first uploaded file
    name = next(iter(uploaded.keys()))
    path = os.path.join("/content", name)
    with open(path, "wb") as f:
        f.write(uploaded[name])
    return path

In [ ]:
print("Upload CSFV PDBs zip (e.g., 'CSFV PDBs.zip'):")
csfv_zip = upload_zip("Upload CSFV zip now")
print("Upload Swine PDBs zip (e.g., 'Swine PDBs.zip'):")
swine_zip = upload_zip("Upload Swine zip now")


Upload CSFV PDBs zip (e.g., 'CSFV PDBs.zip'):
Upload CSFV zip now


Saving CSFV PDBs.zip to CSFV PDBs.zip
Upload Swine PDBs zip (e.g., 'Swine PDBs.zip'):
Upload Swine zip now


Saving Swine PDBs.zip to Swine PDBs (2).zip


In [ ]:
import os
import zipfile

OUT_DIR = "/content"  # replace with your output directory
csfv_zip = "/content/CSFV PDBs.zip"  # path to your CSFV zip
swine_zip = "/content/Swine PDBs.zip"  # path to your Swine zip

# Create folders
csfv_dir = os.path.join(OUT_DIR, "csfv")
swine_dir = os.path.join(OUT_DIR, "swine")
for d in (csfv_dir, swine_dir):
    os.makedirs(d, exist_ok=True)

# Extract Swine zip
try:
    with zipfile.ZipFile(swine_zip, 'r') as z:
        z.extractall(swine_dir)
    print(f"Successfully extracted {swine_zip} to {swine_dir}")
except zipfile.BadZipFile:
    print(f"Error: {swine_zip} is not a valid zip file.")
except FileNotFoundError:
    print(f"Error: {swine_zip} not found.")

# Extract CSFV zip only if it exists
if os.path.exists(csfv_zip):
    try:
        with zipfile.ZipFile(csfv_zip, 'r') as z:
            z.extractall(csfv_dir)
        print(f"Successfully extracted {csfv_zip} to {csfv_dir}")
    except zipfile.BadZipFile:
        print(f"Error: {csfv_zip} is not a valid zip file.")
else:
    print(f"CSFV zip file not found. Please upload a valid zip before extracting.")


Successfully extracted /content/Swine PDBs.zip to /content/swine
Successfully extracted /content/CSFV PDBs.zip to /content/csfv


In [ ]:
# 3)
def find_pdbs(root):
    out=[]
    for dp,dn,fn in os.walk(root):
        for f in fn:
            if f.lower().endswith(".pdb"):
                out.append(os.path.join(dp,f))
    return sorted(out)

csfv_files = find_pdbs(csfv_dir)
swine_files = find_pdbs(swine_dir)
print(f"Found {len(csfv_files)} CSFV pdbs and {len(swine_files)} Swine pdbs")

if len(csfv_files)==0 or len(swine_files)==0:
    raise SystemExit("No pdbs found in an uploaded zip — check your zip contents.")


Found 1 CSFV pdbs and 19 Swine pdbs


In [ ]:

# 4) parse structure and group atoms by residue - return list of residues: each is dict with id, atoms coords, centroid
def parse_structure_residues(path):
    try:
        struct = parser.get_structure(os.path.basename(path), path)
    except Exception as e:
        print("Parse failed:", path, e)
        return []
    residues = []
    for model in struct:
        for chain in model:
            for res in chain:
                if res.get_resname() == "HOH":
                    continue
                if not is_aa(res, standard=True):
                    continue
                atom_coords = []
                for atom in res:
                    try:
                        atom_coords.append(atom.get_coord())
                    except Exception:
                        pass
                if len(atom_coords)==0:
                    continue
                atom_coords = np.array(atom_coords)
                centroid = atom_coords.mean(axis=0)
                # residue_id: (chain, resseq, icode, resname) - small tuple you can use
                resid = (chain.id, res.get_id()[1], res.get_id()[2].strip() if res.get_id()[2] else "", res.get_resname())
                residues.append({
                    "resid": resid,
                    "coords": atom_coords,
                    "centroid": centroid,
                    "n_atoms": atom_coords.shape[0]
                })
    return residues

In [ ]:
# 5) Precompute residues for all swine files and build KDTree of all swine atoms + mapping atom_index->(residue_index)
print("Parsing swine structures and building trees")
swine_store = []  # list of dicts per swine file: {name, path, residues, atom_coords_all, atom_residue_index, tree}
for p in swine_files:
    res_list = parse_structure_residues(p)
    if len(res_list)==0:
        continue
    # flatten atoms
    atom_coords = []
    atom_residx = []  # which residue index each atom belongs to
    for ridx, r in enumerate(res_list):
        for a in r["coords"]:
            atom_coords.append(a)
            atom_residx.append(ridx)
    atom_coords = np.array(atom_coords)
    tree = cKDTree(atom_coords) if atom_coords.shape[0]>0 else None
    swine_store.append({
        "name": os.path.basename(p),
        "path": p,
        "residues": res_list,
        "atom_coords": atom_coords,
        "atom_residx": np.array(atom_residx, dtype=int) if len(atom_residx)>0 else np.array([], dtype=int),
        "tree": tree
    })
print(f"Prepared {len(swine_store)} swine entries")

Parsing swine structures and building trees
Prepared 19 swine entries


In [ ]:
# 6) For each CSFV file, parse residues and compare to each swine file using residue-level contact detection
detailed_rows = []   # will store residue-level contact rows
summary_rows = []    # will store per-protein pair summary (unique contacting residue pairs count)
for cs_p in csfv_files:
    cs_name = os.path.basename(cs_p)
    cs_res = parse_structure_residues(cs_p)
    if len(cs_res)==0:
        print("No residues parsed for", cs_name); continue
    # flatten atoms for cs for KD queries (we will query swine trees with cs atoms)
    cs_atom_coords = []
    cs_atom_residx = []
    for ridx, r in enumerate(cs_res):
        for a in r["coords"]:
            cs_atom_coords.append(a)
            cs_atom_residx.append(ridx)
    cs_atom_coords = np.array(cs_atom_coords)
    cs_atom_residx = np.array(cs_atom_residx, dtype=int)
    # for each swine entry
    for s in swine_store:
        # quick skip if no atoms
        if s["atom_coords"].shape[0]==0 or cs_atom_coords.shape[0]==0:
            continue
        # query swine atom tree using all CSFV atoms: for each CS atom get indices of swine atoms within cutoff
        hits = s["tree"].query_ball_point(cs_atom_coords, r=CUTOFF)
        # build set of contacting residue pairs and also track min distances
        pair_min_dist = {}  # key: (cs_residx, sw_residx) -> min dist
        for cs_atom_idx, sw_atom_idxs in enumerate(hits):
            if len(sw_atom_idxs)==0:
                continue
            cs_ridx = int(cs_atom_residx[cs_atom_idx])
            cs_coord = cs_atom_coords[cs_atom_idx]
            for sw_atom_idx in sw_atom_idxs:
                sw_ridx = int(s["atom_residx"][sw_atom_idx])
                sw_coord = s["atom_coords"][sw_atom_idx]
                d = float(np.linalg.norm(cs_coord - sw_coord))
                key = (cs_ridx, sw_ridx)
                prev = pair_min_dist.get(key)
                if prev is None or d < prev:
                    pair_min_dist[key] = d
        # now convert pair_min_dist to rows (residue-level)
        if len(pair_min_dist)==0:
            continue
        # summary count = # unique residue pairs
        summary_rows.append({
            "CSFV": cs_name,
            "Swine": s["name"],
            "Contacting_Residue_Pairs": len(pair_min_dist)
        })
        # detailed residue rows
        for (cs_ridx, sw_ridx), min_d in pair_min_dist.items():
            cs_resobj = cs_res[cs_ridx]
            sw_resobj = s["residues"][sw_ridx]
            # build human-readable residue identifiers
            cs_res_id = f"{cs_name}|chain={cs_resobj['resid'][0]}|resseq={cs_resobj['resid'][1]}|resname={cs_resobj['resid'][3]}"
            sw_res_id = f"{s['name']}|chain={sw_resobj['resid'][0]}|resseq={sw_resobj['resid'][1]}|resname={sw_resobj['resid'][3]}"
            detailed_rows.append({
                "CSFV": cs_name,
                "Swine": s["name"],
                "CSFV_residue": cs_res_id,
                "Swine_residue": sw_res_id,
                "MinDistance": float(min_d),
                "CSFV_res_centroid_x": float(cs_resobj["centroid"][0]),
                "CSFV_res_centroid_y": float(cs_resobj["centroid"][1]),
                "CSFV_res_centroid_z": float(cs_resobj["centroid"][2]),
                "Swine_res_centroid_x": float(sw_resobj["centroid"][0]),
                "Swine_res_centroid_y": float(sw_resobj["centroid"][1]),
                "Swine_res_centroid_z": float(sw_resobj["centroid"][2])
            })

In [ ]:
# 7) Save detailed residue-level CSV and summary CSV
detailed_df = pd.DataFrame(detailed_rows)
summary_df = pd.DataFrame(summary_rows).sort_values("Contacting_Residue_Pairs", ascending=False)

detailed_path = os.path.join(OUT_DIR, "all_interactions_residue_detailed.csv")
summary_path  = os.path.join(OUT_DIR, "interaction_summary.csv")
detailed_df.to_csv(detailed_path, index=False)
summary_df.to_csv(summary_path, index=False)
print("Saved detailed residue CSV:", detailed_path)
print("Saved summary CSV:", summary_path)

Saved detailed residue CSV: /content/all_interactions_residue_detailed.csv
Saved summary CSV: /content/interaction_summary.csv


In [ ]:
# 8) Choose HIGH pairs (top TOP_PERCENT by Contacting_Residue_Pairs)
if summary_df.empty:
    raise SystemExit("No contacting residue pairs found across dataset.")

k = max(1, int(max(1, len(summary_df)) * TOP_PERCENT))
# use top-k by count to be safe (ensures at least one)
summary_df = summary_df.reset_index(drop=True)
summary_df['rank'] = summary_df['Contacting_Residue_Pairs'].rank(method='first', ascending=False).astype(int)
high_threshold_rank = int(np.ceil(len(summary_df) * TOP_PERCENT))
high_df = summary_df[summary_df['rank'] <= max(1, high_threshold_rank)].copy()
high_path = os.path.join(OUT_DIR, "high_interactions_summary.csv")
high_df.to_csv(high_path, index=False)
print("Saved high interactions summary (top fraction):", high_path, f"(count={len(high_df)})")

Saved high interactions summary (top fraction): /content/high_interactions_summary.csv (count=4)


In [ ]:
# 9) Permutations & combinations from high pairs
pairs = list(high_df[["CSFV","Swine"]].itertuples(index=False, name=None))
if len(pairs) >= 2:
    combs = list(combinations(pairs, 2))
    perms = list(permutations(pairs, 2))
    pd.DataFrame(combs, columns=["Pair1","Pair2"]).to_csv(os.path.join(OUT_DIR,"combinations.csv"), index=False)
    pd.DataFrame(perms, columns=["Pair1","Pair2"]).to_csv(os.path.join(OUT_DIR,"permutations.csv"), index=False)
    print("Saved permutations & combinations CSVs")
else:
    print("Not enough high pairs for permutations/combinations")

Saved permutations & combinations CSVs


In [ ]:
# 10) Heatmap of summary (CSFV x Swine) — pivot and plot (top N labels to keep plot readable)
pivot = summary_df.pivot(index="CSFV", columns="Swine", values="Contacting_Residue_Pairs").fillna(0)
# limit size for readability (take top 40 CSFV and top 40 Swine by totals)
cs_order = summary_df.groupby("CSFV")["Contacting_Residue_Pairs"].sum().sort_values(ascending=False).index.tolist()
sw_order = summary_df.groupby("Swine")["Contacting_Residue_Pairs"].sum().sort_values(ascending=False).index.tolist()
top_cs = cs_order[:40]
top_sw = sw_order[:40]
pivot_small = pivot.reindex(index=top_cs, columns=top_sw).fillna(0)

plt.figure(figsize=(10,8))
plt.imshow(pivot_small, aspect="auto")
plt.colorbar(label="Contacting residue pairs")
plt.xticks(range(len(pivot_small.columns)), pivot_small.columns, rotation=90, fontsize=6)
plt.yticks(range(len(pivot_small.index)), pivot_small.index, fontsize=6)
plt.title("CSFV vs Swine contacting residue pairs (top 40x40)")
heatmap_png = os.path.join(OUT_DIR, "residue_contact_heatmap.png")
plt.tight_layout()
plt.savefig(heatmap_png, dpi=200)
plt.close()
print("Saved heatmap:", heatmap_png)

Saved heatmap: /content/residue_contact_heatmap.png


In [ ]:
# 11) Simple 3D scatter visualization for the top high pair (centroids)
if not high_df.empty:
    top = high_df.iloc[0]
    cs_top = top["CSFV"]; sw_top = top["Swine"]
    # filter detailed rows for that pair
    sel = detailed_df[(detailed_df["CSFV"]==cs_top) & (detailed_df["Swine"]==sw_top)]
    if not sel.empty:
        fig = plt.figure(figsize=(6,6))
        ax = fig.add_subplot(111, projection='3d')
        ax.scatter(sel["CSFV_res_centroid_x"], sel["CSFV_res_centroid_y"], sel["CSFV_res_centroid_z"], s=20, label="CSFV residues")
        ax.scatter(sel["Swine_res_centroid_x"], sel["Swine_res_centroid_y"], sel["Swine_res_centroid_z"], s=20, label="Swine residues")
        ax.set_title(f"Residue centroids: {cs_top} vs {sw_top}")
        ax.legend()
        vis3d_path = os.path.join(OUT_DIR, "residue_centroids_top_pair.png")
        plt.savefig(vis3d_path, dpi=200)
        plt.close()
        print("Saved 3D centroid scatter:", vis3d_path)
    else:
        print("No detailed rows for top pair to plot 3D centroids")


Saved 3D centroid scatter: /content/residue_centroids_top_pair.png


In [ ]:
# 12) Zip results and prepare download
zip_out = os.path.join("/content", "residue_results.zip")
shutil.make_archive(os.path.splitext(zip_out)[0], 'zip', OUT_DIR)
print("Zipped results to:", zip_out)

# Provide download link in Colab
files.download(zip_out)   # will prompt download in Colab
print("Download initiated (if running in Colab). Also check", OUT_DIR)

RuntimeError: File size too large, try using force_zip64

In [ ]:
# Install py3Dmol if not present
!pip install py3Dmol biopython

import py3Dmol
from Bio.PDB import PDBParser
import numpy as np
import os # Import os module

def show_interaction(pdb1_path, pdb2_path, cutoff=5.0):
    parser = PDBParser(QUIET=True)
    st1 = parser.get_structure("prot1", pdb1_path)
    st2 = parser.get_structure("prot2", pdb2_path)

    # Collect atom coords
    atoms1 = [a for a in st1.get_atoms() if a.element != 'H']
    atoms2 = [a for a in st2.get_atoms() if a.element != 'H']
    coords1 = np.array([a.coord for a in atoms1])
    coords2 = np.array([a.coord for a in atoms2])

    # Find interacting residues
    interacting_residues = set()
    for i, c1 in enumerate(coords1):
        dists = np.linalg.norm(coords2 - c1, axis=1)
        if np.any(dists < cutoff):
            interacting_residues.add(atoms1[i].get_parent().get_id())

    # Load in py3Dmol
    view = py3Dmol.view(width=600, height=500)
    with open(pdb1_path) as f:
        view.addModel(f.read(), 'pdb')
    with open(pdb2_path) as f:
        view.addModel(f.read(), 'pdb')

    # Color models
    view.setStyle({'model':0}, {"cartoon": {"color": "cyan"}})
    view.setStyle({'model':1}, {"cartoon": {"color": "magenta"}})

    # Highlight interacting residues (as sticks)
    # Need to handle the tuple residue ID (chain, resseq, icode)
    for res_id in interacting_residues:
        # Construct selection dictionary for py3Dmol
        # py3Dmol selection needs string for resi, and can take chain
        sel = {"model":0, "chain":res_id[0], "resi":str(res_id[1])}
        view.addStyle(sel, {"stick": {"color": "yellow"}})


    view.zoomTo()
    return view


# Example: show one CSFV protein vs one swine protein
# Replace with actual paths from your extracted files
# You can inspect the /content/simple_residue_results/csfv and /content/simple_residue_results/swine directories
# to find appropriate PDB files.
csfv_example_pdb = csfv_files[0] # Using the first found CSFV file
swine_example_pdb = swine_files[0] # Using the first found Swine file


view = show_interaction(csfv_example_pdb, swine_example_pdb, cutoff=5.0)
view.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 9.1 MB/s eta 0:00:00
